# 02 — Bottle Correction
Merge all drift-corrected deployments for the reference designator, then
apply the shipboard discrete (bottle) offset correction deployment by
deployment, following Palevsky et al. (2023).

In [ ]:
import sys, glob
import numpy as np
import pandas as pd
import xarray as xr
import yaml
sys.path.insert(0, '..')

from nitrate import bottles

In [ ]:
config = yaml.safe_load(open('../config/GI01SUMO-SBD11-08-NUTNRB000.yaml'))
refdes = config['refdes']
site, node, sensor = refdes.split('-', 2)
data_dir = f"../{config['paths']['data_dir']}"

## 1. Merge the drift-corrected deployments

In [ ]:
drift_files = sorted(glob.glob(f'{data_dir}{refdes}_deployment*_drift_corrected.nc'))
data = None
for f in drift_files:
    ds = xr.open_dataset(f)
    data = ds if data is None else xr.concat([data, ds], dim='time')
data

## 2. Load shipboard discrete nitrate samples

In [ ]:
if config.get('bottle_csv'):
    bottle_data = pd.read_csv(config['bottle_csv'])
else:
    bottle_data = pd.read_csv(f'{data_dir}cleaned_bottle_data.csv')

bottle_data['Cruise'] = bottle_data['Cruise'].apply(bottles.remove_last_letter)
bottle_data['Start Time [UTC]'] = bottle_data['Start Time [UTC]'].astype(str).str.strip('Z')
bottle_data['Start Time [UTC]'] = bottle_data['Start Time [UTC]'].astype('datetime64[ns]')
bottle_data

## 3. Look up deployment/recovery cruise IDs

In [ ]:
deployments = np.unique(data['deployment'].values).astype(int).tolist()
deploy_info = pd.DataFrame(bottles.get_deployment_info(site, node, sensor, deployments))
deploy_info.set_index(keys='deploymentNumber', drop=True, inplace=True)
deploy_info

## 4. Apply the bottle correction, deployment by deployment

In [ ]:
corrected = None
smoothed = None
deploy_bottles = None
recover_bottles = None
for depNum in deployments:
    depdata = data.where(data.deployment == depNum, drop=True)
    depdata, smoothed_data, depBottles, recBottles = bottles.bottle_correction(depdata, deploy_info, bottle_data)

    corrected = depdata.copy(deep=True) if corrected is None else xr.concat([corrected, depdata], dim='time')
    smoothed = smoothed_data.copy(deep=True) if smoothed is None else xr.concat([smoothed, smoothed_data], dim='time')

    depBottles = depBottles.copy(); depBottles['Deployment'] = depNum
    recBottles = recBottles.copy(); recBottles['Deployment'] = depNum
    deploy_bottles = depBottles if deploy_bottles is None else pd.concat([deploy_bottles, depBottles])
    recover_bottles = recBottles if recover_bottles is None else pd.concat([recover_bottles, recBottles])

corrected

## 5. Plot one deployment against the bottle observations

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

depNum = deployments[0]
depdata = corrected.where(corrected.deployment == depNum, drop=True)
dmask = deploy_bottles['Deployment'] == depNum
rmask = recover_bottles['Deployment'] == depNum

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(depdata['time'], depdata['corrected_nitrate_concentration'], marker='.', linestyle='',
        color='tab:blue', label='T-S(-P) Corrected')
ax.plot(depdata['time'], depdata['drift_corrected_nitrate'], marker='.', linestyle='',
        color='tab:green', label='+ Drift Corrected')
ax.plot(depdata['time'], depdata['bottle_corrected_nitrate'], marker='.', linestyle='',
        color='tab:orange', label='+ Bottle Corrected')
ax.plot(deploy_bottles[dmask].index, deploy_bottles[dmask]['Discrete Nitrate [uM]'], marker='o',
        linestyle='', color='tab:red', markeredgecolor='black', markersize=8, label='Discrete Bottle Sample')
ax.plot(recover_bottles[rmask].index, recover_bottles[rmask]['Discrete Nitrate [uM]'], marker='o',
        linestyle='', color='tab:red', markeredgecolor='black', markersize=8)
ax.grid()
ax.legend()
ax.set_ylabel('Nitrate Concentration [uM]')
ax.set_title(f'{refdes} — Deployment {depNum}')
fig.autofmt_xdate()

## 6. Save the bottle-corrected dataset

In [ ]:
outpath = f'{data_dir}{refdes}_bottle_corrected.nc'
corrected.to_netcdf(outpath, format='netcdf4', engine='h5netcdf')
print(f'Saved -> {outpath}')